This data cleaning script eliminates duplicates, removes low-quality data, and exports a structured dataset that can be used to support sentiment analysis down the line.


In [1]:
import pandas as pd
import re

input_file = "sony_raw_data.csv"
output_file = "sony_cleaned_data.csv"

In [2]:
df = pd.read_csv(input_file)

print("Shape of raw data:", df.shape)
print("\nColumns:")
print(df.columns.tolist())

print("\nFirst 5 rows:")
print(df.head())

Shape of raw data: (104, 5)

Columns:
['product', 'source', 'url', 'paragraph_id', 'raw_text']

First 5 rows:
             product     source                                 url  \
0  Sony VAIO Laptops  Wikipedia  https://en.wikipedia.org/wiki/VAIO   
1  Sony VAIO Laptops  Wikipedia  https://en.wikipedia.org/wiki/VAIO   
2  Sony VAIO Laptops  Wikipedia  https://en.wikipedia.org/wiki/VAIO   
3  Sony VAIO Laptops  Wikipedia  https://en.wikipedia.org/wiki/VAIO   
4  Sony VAIO Laptops  Wikipedia  https://en.wikipedia.org/wiki/VAIO   

   paragraph_id                                           raw_text  
0             2  VAIO Corporation ( VAIO 株式会社 , Baio Kabushiki ...  
1             3  Vaio began as a brand of Sony , [ 5 ] introduc...  
2             4  In 2025, JIP completed the sale of its 91.40% ...  
3             5  Originally an acronym of "Video Audio Input Ou...  
4             6  As of 2023, Vaio is operational in the followi...  


In [3]:
df.columns = df.columns.str.strip().str.lower().str.replace(" ", "_")

print("\nStandardized columns:")
print(df.columns.tolist())


Standardized columns:
['product', 'source', 'url', 'paragraph_id', 'raw_text']


In [4]:
text_columns = ["product", "source", "url", "paragraph_id", "raw_text"]

for col in text_columns:
    df[col] = df[col].astype(str).str.strip()

In [5]:
product_map = {
    "sony vaio laptops": "Sony VAIO Laptops",
    "sony playstation vue": "Sony PlayStation Vue",
    "sony playmemories online": "Sony PlayMemories Online"
}

df["product"] = df["product"].str.lower().map(product_map).fillna(df["product"])

In [6]:
df["source"] = df["source"].str.strip()

source_map = {
    "wikipedia": "Wikipedia",
    "the verge": "The Verge",
    "sony support": "Sony Support",
    "sony help guide": "Sony Help Guide",
    "britannica laptop computer": "Britannica Laptop Computer",
    "britannica streaming media": "Britannica Streaming Media"
}

df["source"] = df["source"].str.lower().map(source_map).fillna(df["source"])

In [7]:

df["url"] = df["url"].str.strip().str.lower()
def clean_text(text):
    text = str(text)

    # Remove line breaks and tabs
    text = re.sub(r"[\r\n\t]+", " ", text)

    # Remove multiple spaces
    text = re.sub(r"\s+", " ", text)

    # Remove citation-style bracket artifacts like [ 5 ] or [5]
    text = re.sub(r"\[\s*\d+\s*\]", "", text)

    # Remove extra spaces again after bracket cleanup
    text = re.sub(r"\s+", " ", text)

    return text.strip()

df["clean_text"] = df["raw_text"].apply(clean_text)

In [8]:
df = df[df["clean_text"].notna()]
df = df[df["clean_text"].str.len() > 20]

print("\nShape after removing weak text rows:", df.shape)


Shape after removing weak text rows: (104, 6)


In [9]:
df["char_count"] = df["clean_text"].str.len()
df["word_count"] = df["clean_text"].str.split().str.len()

print("\nSummary of text length features:")
print(df[["char_count", "word_count"]].describe())


Summary of text length features:
        char_count  word_count
count   104.000000  104.000000
mean    491.230769   78.711538
std     362.778363   57.523093
min      59.000000   10.000000
25%     174.000000   30.000000
50%     425.500000   68.500000
75%     719.750000  118.750000
max    1713.000000  276.000000


In [10]:
before_dedup = df.shape[0]

df = df.drop_duplicates(subset=["product", "source", "url", "clean_text"])

after_dedup = df.shape[0]

print(f"\nDuplicates removed: {before_dedup - after_dedup}")
print("Shape after deduplication:", df.shape)


Duplicates removed: 0
Shape after deduplication: (104, 8)


In [ ]:
df["is_short_text"] = df["word_count"] < 15

df["is_support_source"] = df["source"].isin(["Sony Support", "Sony Help Guide"])

print("\nFlag counts:")
print("Short text rows:", df["is_short_text"].sum())
print("Support/help rows:", df["is_support_source"].sum())

In [11]:
df["paragraph_id"] = df["paragraph_id"].astype(str).str.strip()

In [13]:
df["is_short_text"] = df["word_count"] < 15
df["is_support_source"] = df["source"].isin(["Sony Support", "Sony Help Guide"])

final_columns = [
    "product",
    "source",
    "url",
    "paragraph_id",
    "raw_text",
    "clean_text",
    "char_count",
    "word_count",
    "is_short_text",
    "is_support_source"
]

df = df[final_columns]

In [14]:
df.to_csv(output_file, index=False)

print(f"\nCleaned file saved as: {output_file}")
print("Final shape:", df.shape)
print("\nFinal preview:")
print(df.head())


Cleaned file saved as: sony_cleaned_data.csv
Final shape: (104, 10)

Final preview:
             product     source                                 url  \
0  Sony VAIO Laptops  Wikipedia  https://en.wikipedia.org/wiki/vaio   
1  Sony VAIO Laptops  Wikipedia  https://en.wikipedia.org/wiki/vaio   
2  Sony VAIO Laptops  Wikipedia  https://en.wikipedia.org/wiki/vaio   
3  Sony VAIO Laptops  Wikipedia  https://en.wikipedia.org/wiki/vaio   
4  Sony VAIO Laptops  Wikipedia  https://en.wikipedia.org/wiki/vaio   

  paragraph_id                                           raw_text  \
0            2  VAIO Corporation ( VAIO 株式会社 , Baio Kabushiki ...   
1            3  Vaio began as a brand of Sony , [ 5 ] introduc...   
2            4  In 2025, JIP completed the sale of its 91.40% ...   
3            5  Originally an acronym of "Video Audio Input Ou...   
4            6  As of 2023, Vaio is operational in the followi...   

                                          clean_text  char_count  word_co